# Assignment 2: BRFSS Diabetes Classification
**Temirlan Sapargali | SE-2403**

Predict reported diabetes (1) versus source class zero (0), excluding prediabetes.
Run all cells. In Colab, upload `diabetes_012_health_indicators_BRFSS2015.csv` when asked.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    classification_report, roc_auc_score, average_precision_score,
    ConfusionMatrixDisplay)


## Load Dataset


In [ ]:
path = Path("data/diabetes_012_health_indicators_BRFSS2015.csv")
if not path.exists():
    path = Path(path.name)
if not path.exists():
    from google.colab import files
    path = Path(next(iter(files.upload())))
data = pd.read_csv(path)
print("Source shape:", data.shape)
print(data["Diabetes_012"].value_counts().sort_index())
data.head()


## Data Preprocessing
Exclude prediabetes. BRFSS zeros are valid; no zero replacement or scaling is needed for Random Forest.


In [ ]:
data = data.loc[data["Diabetes_012"].isin([0, 2])].copy()
X = data.drop(columns="Diabetes_012").astype("int16")
y = data["Diabetes_012"].eq(2).astype(int)
print("Prepared shape:", X.shape, "Missing cells:", X.isna().sum().sum())
print("Classes:\n", y.value_counts().sort_index())
print("Repeated complete rows:", data.duplicated().sum())


In [ ]:
# About 80/20; identical predictor profiles stay in the same partition.
groups = pd.util.hash_pandas_object(X, index=False)
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train, test = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train], X.iloc[test]
y_train, y_test = y.iloc[train], y.iloc[test]
assert set(groups.iloc[train]).isdisjoint(groups.iloc[test])
print("Train:", len(train), "Test:", len(test))

classifier = RandomForestClassifier(n_estimators=100, min_samples_leaf=10,
    class_weight="balanced_subsample", n_jobs=-1, random_state=42)
classifier.fit(X_train, y_train)


## Random Forest Results


In [ ]:
pred = classifier.predict(X_test)
prob = classifier.predict_proba(X_test)[:, 1]
print(pd.Series({
    "Accuracy": accuracy_score(y_test, pred),
    "All-negative accuracy": 1 - y_test.mean(),
    "Balanced accuracy": balanced_accuracy_score(y_test, pred),
    "ROC-AUC": roc_auc_score(y_test, prob),
    "Average precision": average_precision_score(y_test, prob)
}).round(3))
print(classification_report(y_test, pred, digits=3, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Confusion matrix")
plt.show()


In [ ]:
importance = pd.Series(classifier.feature_importances_, index=X.columns).sort_values()
importance.plot.barh(title="Feature importance", figsize=(8, 6))
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


**For Word:** dataset preview, preprocessing checks, metrics/confusion matrix, feature importance.
This is a fixed Random Forest baseline. Survey labels do not confirm diabetes type or future onset.
